# CUDA waste-detection prototype - full rerun (includes fine-tune)

**Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**.

A fresh clone does not contain the 313 MB dataset zip. Choose ONE of these
before running:

1. Paste the working Roboflow export URL into the next cell, or
2. Upload `zerowaste_yolov8.zip` to `data/raw/` with the Files sidebar.

The runner archives old CSVs, rebuilds the Phase A kernels, fine-tunes
YOLOv8n on the ZeroWaste subset, re-measures every stage on real frames,
and regenerates the 11 slide images. Failures are written to
`results/failures.md`; missing measurements are labelled `NOT RUN`, never
estimated. This is a prototype, not a deployed detector.


In [ ]:
# ---- Bootstrap: find or clone the repo, stream commands live ----------------
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").is_dir()
GIT_URL = "https://github.com/pkbros/CUDA-accelerated-waste-detection-pipeline.git"


def find_repo_root(start=None):
    here = Path(start or Path.cwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "include" / "bench_common.h").exists():
            return cand
    return None


REPO = find_repo_root()
if REPO is None and IN_COLAB:
    for cand in Path("/content").glob("*/include/bench_common.h"):
        REPO = cand.parent.parent
        break
if REPO is None and IN_COLAB:
    target = Path("/content/cuda-waste-pipeline")
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(target)], check=True)
    REPO = find_repo_root(target)
if REPO is None:
    raise SystemExit("Repo not found: include/bench_common.h is missing.")

# Always execute the latest committed code; untracked files (the zip, weights)
# survive this. Only do it in Colab, never in a local checkout.
if IN_COLAB and (REPO / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "-q", "origin", "main"], check=False
    )
    subprocess.run(
        ["git", "-C", str(REPO), "reset", "--hard", "-q", "origin/main"], check=False
    )

os.chdir(REPO)
sys.path.insert(0, str(REPO / "phase_b"))
for folder in ("results", "build", "reports", "weights"):
    (REPO / folder).mkdir(exist_ok=True)


def sh(cmd, check=True):
    print(f"$ {cmd}", flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        cmd, shell=True, cwd=str(REPO), env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    chunks = []
    for line in proc.stdout:
        chunks.append(line)
        print(line, end="", flush=True)
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {cmd}")
    return proc.returncode, "".join(chunks)


def record_failure(stage, detail):
    path = REPO / "results" / "failures.md"
    new = not path.exists()
    with path.open("a", encoding="utf-8") as fh:
        if new:
            fh.write("# Recorded failures\n\nPrototype run; failures are kept deliberately.\n\n")
        fh.write(f"- {stage}: {str(detail).strip()[:400]}\n")
    print(f"[recorded failure] {stage}: {str(detail).strip()[:200]}")


print("repo:", REPO)
print("colab:", IN_COLAB)


In [ ]:
# @title 1. Dataset source (paste URL OR just upload the zip)
# @markdown If you uploaded `zerowaste_yolov8.zip` with the Files sidebar,
# @markdown leave the URL blank - the notebook finds and copies it for you.
# @markdown The URL contains a `key=` token and is set only in this runtime,
# @markdown never written to the repo.
EXPORT_URL = ""  # @param {type:"string"}
if EXPORT_URL.strip():
    os.environ["ROBOFLOW_EXPORT_URL"] = EXPORT_URL.strip()

import re
import shutil

dest = REPO / "data/raw/zerowaste_yolov8.zip"
if not dest.exists() and IN_COLAB:
    uploaded = [
        f for f in Path("/content").rglob("*.zip")
        if re.search(r"zero.?waste|yolov8", f.name, re.I) and f.resolve() != dest.resolve()
    ]
    if uploaded:
        src = max(uploaded, key=lambda f: f.stat().st_size)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest)
        print(f"copied uploaded zip: {src.name} -> {dest}")

print("export URL configured:", bool(os.environ.get("ROBOFLOW_EXPORT_URL")))
print("local zip present:", dest.exists(), dest.stat().st_size / 1e6 if dest.exists() else "")


In [ ]:
# ---- 2. Run the whole measured pipeline -----------------------------------
import torch

assert torch.cuda.is_available(), "No GPU: set Runtime > T4 GPU and reconnect."
print("gpu:", torch.cuda.get_device_name(0), "torch:", torch.__version__)
rc, _ = sh(f"python {shlex.quote(str(REPO / 'demo/run_colab_demo.py'))}", check=False)
print("runner exit code:", rc)


In [ ]:
# ---- 3. Read the summary and the two most important slide images -----------
from IPython.display import Image, Markdown, display

summary = REPO / "results/summary.md"
if summary.exists():
    display(Markdown(summary.read_text(encoding="utf-8")))
else:
    print("summary.md not written (see failures.md)")

for name in ("03_latency_breakdown.png", "11_detection_quality.png"):
    path = REPO / "docs/ppt_pack" / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))


In [ ]:
# ---- 4. Ship the numbers back ---------------------------------------------
# results_bundle.zip downloads from the Colab web UI. The inline dump below is
# saved into this .ipynb file too, so the numbers survive even without a
# browser download (e.g. under the Colab VS Code extension).
import zipfile

results = REPO / "results"
bundle = results / "results_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(results.rglob("*")):
        if path.is_file() and path.suffix != ".zip":
            zf.write(path, path.relative_to(REPO))
print(f"wrote {bundle} ({bundle.stat().st_size / 1024:.1f} KiB)")

if IN_COLAB:
    try:
        from google.colab import files

        files.download(str(bundle))
    except Exception as exc:
        print(f"browser download unavailable ({type(exc).__name__}: {exc})")

for path in sorted(results.glob("*.csv")):
    print("=" * 70)
    print(f"FILE results/{path.name}")
    print("=" * 70)
    print(path.read_text(encoding="utf-8").rstrip())
    print()

failures = results / "failures.md"
if failures.exists():
    print("=" * 70)
    print("FILE results/failures.md")
    print("=" * 70)
    print(failures.read_text(encoding="utf-8").rstrip())


## What changed in this run

- **Fine-tune first**: YOLOv8n is trained 30 epochs on the ZeroWaste subset
  (`data/waste.yaml`); `results/finetune.csv` records the measured mAP.
- **One model everywhere**: the tuned checkpoint is exported to ONNX and used
  by the inference ladder, per-frame timings and the multi-feed pipeline.
- **Real frames**: batches 1/4/8 and all demo timings use val images from the
  dataset, not a zero tensor. `results/pipeline_agreement.csv` checks that
  optimizing the pipeline did not change the detections.
- **Fresh data only**: previous CSVs are archived under `results/archive_*`.
- **No video render**: the deliverable is the slide-image pack in
  `docs/ppt_pack/`, not an offline demo clip.

If a stage fails (for example `ncu` counters on a restricted runtime), the
reason is in `results/failures.md` and the affected panel reads `NOT RUN`.
